In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 1

In [4]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


Rows: 2000
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


### Question -1

In [5]:
answer_counts = df["answer"].value_counts()

print("\nQ1")
print(answer_counts)

most_freq = answer_counts.max()
least_freq = answer_counts.min()

q1_answer = most_freq + least_freq

print("Q1 Answer =", q1_answer)


Q1
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Q1 Answer = 814


### Question - 2

In [6]:
# CLEANING FUNCTION


translator = str.maketrans("", "", string.punctuation)

def clean_text(text):
    text = str(text).lower()
    text = text.translate(translator)
    return text

cleaned_prompts = df["prompt"].apply(clean_text)

vocab = set()

for text in cleaned_prompts:
    vocab.update(text.split())

q2_answer = len(vocab)

print("\nQ2 Answer =", q2_answer)


Q2 Answer = 859


### Question - 3 

In [7]:
row1_prompt = clean_text(df.iloc[0]["prompt"])

tokens = row1_prompt.split()

filtered_tokens = [
    word for word in tokens
    if word not in ENGLISH_STOP_WORDS
]

q3_answer = len(filtered_tokens)

print("\nQ3 Answer =", q3_answer)


Q3 Answer = 13


### Question - 4 

In [8]:
combined_text = (
    df["prompt"].fillna("") + " " +
    df["A"].fillna("") + " " +
    df["B"].fillna("") + " " +
    df["C"].fillna("") + " " +
    df["D"].fillna("") + " " +
    df["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")

vectorizer.fit(combined_text)

q4_answer = len(vectorizer.get_feature_names_out())

print("\nQ4 Answer =", q4_answer)


Q4 Answer = 2762


### Question - 5

In [9]:
prompt_vec = vectorizer.transform([df.iloc[0]["prompt"]])
option_a_vec = vectorizer.transform([df.iloc[0]["A"]])

sim = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print("\nQ5 Answer =", round(sim, 4))


Q5 Answer = 0.272


### Question - 6

In [10]:
correct = 0

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        score = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores[option] = score

    pred = max(scores, key=scores.get)

    if pred == row["answer"]:
        correct += 1

q6_answer = (correct / len(df)) * 100

print("\nQ6 Answer =", round(q6_answer, 2))


Q6 Answer = 13.55


### Question - 7

In [11]:
# MAP@3 FUNCTION

def apk(actual, predicted, k=3):

    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1)

    return 0.0

def mapk(actuals, predictions, k=3):

    scores = [
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ]

    return np.mean(scores)


q7_answer = apk(
    "C",
    ["C", "A", "B"],
    3
)

print("\nQ7 Answer =", q7_answer)


Q7 Answer = 1.0


### Question - 8

In [12]:
q8_answer = apk(
    "B",
    ["D", "B", "E"],
    3
)

print("\nQ8 Answer =", q8_answer)


Q8 Answer = 0.5


### Question - 9 

In [13]:
ranking = (
    df["answer"]
    .value_counts()
    .index
    .tolist()
)

top3_majority = ranking[:3]

predictions = [
    top3_majority
    for _ in range(len(df))
]

q9_answer = mapk(
    df["answer"].tolist(),
    predictions,
    k=3
)

print("\nQ9 Answer =", round(q9_answer, 5))



Q9 Answer = 0.42125


### Question - 10

In [14]:
predictions = []

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = []

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores.append((option, sim))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [x[0] for x in scores[:3]]

    predictions.append(top3)

q10_answer = mapk(
    df["answer"].tolist(),
    predictions,
      k=3
)

print("\nQ10 Answer =", round(q10_answer, 6))


Q10 Answer = 0.296167
